## Step 1: Install Required Libraries

This cell installs the specific versions of all libraries needed for LoRA fine tuning:

- `transformers`: Hugging Face core library for loading and training language models
- `datasets`: for loading and splitting the training data into train, validation and test sets
- `peft`: Parameter Efficient Fine Tuning library that implements LoRA (Low Rank Adaptation)
- `trl`: Transformer Reinforcement Learning library that provides SFTTrainer for supervised fine tuning
- `accelerate`: enables distributed and mixed precision training across multiple GPUs
- `bitsandbytes`: enables 4 bit quantization (QLoRA) to dramatically reduce GPU memory usage
- `mlflow`: Red Hat OpenShift AI compatible fork for experiment tracking and model lineage

Pinning exact versions ensures reproducibility across different environments. The mlflow package is installed from a custom Red Hat fork that is compatible with the RHOAI platform.

In [1]:
!pip install transformers==4.57.1 datasets==4.0.0 peft==0.17.0 trl==0.21.0 accelerate==1.10.0 bitsandbytes==0.48.1
import os

!pip install "git+https://github.com/red-hat-data-services/mlflow@rhoai-3.3"


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/red-hat-data-services/mlflow (to revision rhoai-3.3) to /tmp/pip-req-build-xm2sduqp
  Running command git clone --filter=blob:none --quiet https://github.com/red-hat-data-services/mlflow /tmp/pip-req-build-xm2sduqp
  Running command git checkout -b rhoai-3.3 --track origin/rhoai-3.3
  Switched to a new branch 'rhoai-3.3'
  branch 'rhoai-3.3' set up to track 'origin/rhoai-3.3'.
  Resolved https://github.com/red-hat-data-services/mlflow to commit c0f86528f988088bc07be8acc0db50074acee0f4
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 2: Configure MLflow Environment Variables

MLflow is used to track experiments, log parameters and record dataset lineage during training. This cell sets the environment variables required to connect to a remote MLflow tracking server running on OpenShift AI.

- `MLFLOW_TRACKING_URI`: the endpoint of the MLflow server inside the cluster
- `MLFLOW_TRACKING_TOKEN`: authentication token for secure access
- `MLFLOW_WORKSPACE` and `MLFLOW_EXPERIMENT_NAME`: logical grouping for this fine tuning experiment
- `MLFLOW_RUN_NAME`: a human readable name for this specific training run
- `MLFLOW_TRACKING_HEADERS`: HTTP headers sent with every tracking API call, including the workspace identifier
- `MLFLOW_TRACKING_INSECURE_TLS`: set to "true" to allow self signed certificates in cluster environments

These variables are picked up automatically by the MLflow client and the Hugging Face Trainer integration, so no additional configuration is needed in the training code.

In [2]:
import os

os.environ["MLFLOW_TRACKING_TOKEN"] = "sha256~ZIta0yVdVcziDRD0JDwPKIksGXAaP6DEL8cwluSJs_U"
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
os.environ["MLFLOW_WORKSPACE"] = "intent-classification-sid"
os.environ["MLFLOW_RUN_NAME"] = "decoder-sft-run"
os.environ["MLFLOW_TRACKING_HEADERS"] = "{\"X-MLFLOW-WORKSPACE\": \"intent-classification-sid\", \"Content-Type\": \"application/json\"}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "decoder-sft"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
print("Environment variables set!")

Environment variables set!


## Step 3: Import Libraries

This cell imports all the modules used throughout the notebook:

- `torch`: PyTorch, the deep learning framework underlying all model operations
- `AutoModelForCausalLM` and `AutoTokenizer`: load pretrained language models and their tokenizers from the Hugging Face Hub
- `BitsAndBytesConfig`: configures 4 bit quantization parameters for QLoRA
- `LoraConfig` (from peft): defines the LoRA adapter architecture including rank, alpha and target modules
- `SFTTrainer` and `SFTConfig` (from trl): supervised fine tuning trainer and its configuration class
- `load_dataset`: loads training data from JSON files into a Hugging Face Dataset object
- `mlflow`: experiment tracking client for logging parameters, metrics and dataset lineage
- `TrainerCallback`: base class for creating custom callbacks that hook into the Hugging Face training loop

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import shutil
import os
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from datetime import datetime
import json
import mlflow
from transformers import TrainerCallback

/opt/app-root/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 4: Define MLflow Lineage Callback

This cell defines a custom training callback called `MLflowLineageCallback`. Hugging Face Trainer already creates an MLflow run and logs standard training metrics automatically. This callback extends that by injecting additional metadata into the same run:

- Model name and data path as MLflow parameters
- LoRA hyperparameters (rank, alpha, dropout) for full reproducibility
- Dataset lineage: the train, validation and test splits are converted to MLflow dataset objects and logged with their S3 source URI

The `on_train_begin` hook fires once at the start of training. The `is_world_process_zero` guard ensures this code only runs on the main process during distributed training, preventing duplicate logging across GPUs.

In [4]:
class MLflowLineageCallback(TrainerCallback):
    """
    custom callback to inject S3 dataset lineage and LoRA hyperparameters 
    into the MLflow run automatically created by Hugging Face Trainer
    """
    def __init__(self, custom_args, peft_config, train_ds, val_ds, test_ds):
        self.custom_args = custom_args
        self.peft_config = peft_config
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.test_ds = test_ds

    def on_train_begin(self, args, state, control, **kwargs):
        # only execute on the main process (Rank 0) 
        if state.is_world_process_zero:
            print("Injecting S3 data lineage into Hugging Face's active MLflow run...")
            
            # HF automatically logs learning_rate, batch_size, epochs, etc.
            # so we omit them here to prevent MLflow "duplicate param" errors
            mlflow.log_params({
                "model_name": self.custom_args.model_name,
                "data_path": self.custom_args.data_path,
                "lora_r": self.peft_config.r,
                "lora_alpha": self.peft_config.lora_alpha,
                "lora_dropout": self.peft_config.lora_dropout,
            })
            
            # convert datasets and log them directly into HF's run
            ds_train = mlflow.data.from_pandas(
                self.train_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_train"
            )
            ds_val = mlflow.data.from_pandas(
                self.val_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_val"
            )
            ds_test = mlflow.data.from_pandas(
                self.test_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_test"
            )
            
            mlflow.log_input(ds_train, context="training")
            mlflow.log_input(ds_val, context="evaluation")
            mlflow.log_input(ds_test, context="test")

## Step 5: Define the Data Formatting Function

This function converts each training example from the enterprise data schema into a standardized chat message format that SFTTrainer expects.

The function handles three key tasks:

1. System prompt injection: adds the intent classifier instruction as a system message. Not all models support the system role (for example, Gemma does not), so the function checks the model name and either adds a system message or prepends the instruction to the first user message.

2. Session history mapping: iterates over the conversation history and maps each turn to the appropriate chat role (user or assistant).

3. Target label formatting: appends the ground truth intent as an assistant response in JSON format. This is the target output that the model learns to generate during training. Without this final assistant message, the model would have no signal for what it should produce.

The output is a dictionary with a single `messages` key containing the list of chat turns. This is the standard input format expected by SFTTrainer when using chat templates.

In [5]:
def format_messages_for_training(row: dict, base_model_id: str) -> dict:
    """
    translates the enterprise data schema into a model agnostic chat template for SFTTrainer
    """
    messages = []
    system_prompt = (
        "You are an AI intent classifier. Analyze the conversation history "
        "and the latest user message. You must output only a valid JSON object "
        "containing the predicted 'intent'."
    )
    
    model_name = base_model_id.lower()
    
    # handle model-specific system prompt support
    if "qwen" in model_name or "phi" in model_name or "deepseek" in model_name:
        messages.append({"role": "system", "content": system_prompt})
    elif "gemma" in model_name:
        pass 
    else:
        messages.append({"role": "system", "content": system_prompt})

    # process session history
    for turn in row.get("session_history", []):
        print (turn)
        hf_role = "assistant" if turn.get("role") == "assistant" else "user"
        messages.append({"role": hf_role, "content": turn.get("content", "")})
        
    # process current message
    current_msg = row.get("message", "")
    if "gemma" in model_name and len(messages) == 0:
        current_msg = f"{system_prompt}\n\n{current_msg}"
        
    messages.append({"role": "user", "content": current_msg})
    
    # critical for training: append the target output so the model learns to generate it
    target_output = json.dumps({"intent": row.get("intent", "UNKNOWN")})
    messages.append({
        "role": "assistant",
        "content": target_output
    })
    
    return {"messages": messages}

## Step 6: Test the Formatting Function

Before applying the formatter to the full dataset, this cell validates it with sample data to catch issues early.

Two scenarios are tested:

1. Qwen model with session history: verifies that the system prompt, history turns and target intent are all present in the correct order.

2. Gemma model with no session history: verifies the fallback behavior where the system prompt is prepended to the first user message instead of appearing as a separate system role.

Check the printed output to confirm that each message has the expected role and content before proceeding to the actual training data.

In [6]:
# test the formatter with a sample row
sample_row = {
    "message": "I want to change my shipping address",
    "intent": "UPDATE_SHIPPING",
    "session_history": [
        {"role": "assistant", "content": "Hello! How can I help you today?"},
        {"role": "user", "content": "I placed an order yesterday"}
    ]
}

# test with Qwen (supports system prompt)
result = format_messages_for_training(sample_row, "Qwen/Qwen2.5-7B-Instruct")
print("=== Qwen formatted output ===")
for msg in result["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}")

print()

# test with Gemma (no system prompt, prepended to first user message)
sample_no_history = {"message": "check my balance", "intent": "CHECK_BALANCE", "session_history": []}
result_gemma = format_messages_for_training(sample_no_history, "google/gemma-2b")
print("=== Gemma formatted output (no history) ===")
for msg in result_gemma["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}")

{'role': 'assistant', 'content': 'Hello! How can I help you today?'}
{'role': 'user', 'content': 'I placed an order yesterday'}
=== Qwen formatted output ===
  [system]: You are an AI intent classifier. Analyze the conversation history and the latest
  [assistant]: Hello! How can I help you today?
  [user]: I placed an order yesterday
  [user]: I want to change my shipping address
  [assistant]: {"intent": "UPDATE_SHIPPING"}

=== Gemma formatted output (no history) ===
  [user]: You are an AI intent classifier. Analyze the conversation history and the latest
  [assistant]: {"intent": "CHECK_BALANCE"}


## Step 7: Set Training Configuration

This cell defines the core configuration variables for the fine tuning run:

- `model_name`: the base model to fine tune (Qwen2.5 3B Instruct in this case)
- `data_path`: local path to the training data in JSON format
- `output_dir`: where the trained LoRA adapter weights will be saved
- `dataset_source`: S3 URI recorded in MLflow for data lineage tracking
- `run_name`: identifier for this training run in MLflow

The cell also handles multi GPU setup by reading the `LOCAL_RANK` environment variable, which is set automatically by torchrun during distributed training. For single GPU execution it defaults to 0.

On rank 0 (the main process), the output directory is cleaned to prevent corruption from stale adapter weights left over from previous runs.

In [7]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
data_path = "./mnt/data/train.json"      # path to JSON training data
output_dir = "./mnt/data/adapters"
dataset_source = ""                               # s3 uri for MLflow lineage
run_name = "lora-finetune-run"

# multi GPU rank: set by torchrun, defaults to 0 for single GPU
local_rank = int(os.environ.get("LOCAL_RANK", 0))
print(f"local_rank: {local_rank}")

# rank 0 housekeeping: clean old output to avoid adapter corruption
if local_rank == 0:
    if os.path.exists(output_dir):
        print(f"cleaning existing output dir: {output_dir}")
        shutil.rmtree(output_dir)

local_rank: 0
cleaning existing output dir: ./mnt/data/adapters


## Step 8: Load Model, Prepare Data and Train

This is the main training cell that orchestrates the entire fine tuning pipeline. It performs the following steps in order:

### 8a: Quantization Configuration (QLoRA)

BitsAndBytesConfig is used to load the model in 4 bit precision using the NF4 (Normal Float 4) quantization scheme with double quantization enabled. This reduces GPU memory usage from roughly 6 GB (fp16) to under 2 GB for a 3B parameter model, making it possible to fine tune on a single T4 GPU.

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"quantization config: {bnb_config}")
print(f"loading model: {model_name}...")

quantization config: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

loading model: Qwen/Qwen2.5-7B-Instruct...


### 8b: Model Loading

The pretrained model is loaded with the quantization config applied. The torch dtype is explicitly set to float16 and use_cache is disabled (required when gradient checkpointing is active during training).

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, 
    device_map={"":local_rank},
    torch_dtype=torch.bfloat16, 
    trust_remote_code=True 
)

# CRITICAL: Force the base config to fp16 so PEFT doesn't secretly initialize LoRA weights in bfloat16
#model.config.torch_dtype = torch.float16
model.config.torch_dtype = torch.bfloat16
model.config.use_cache = False

`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

### 8c: Tokenizer Setup

The tokenizer is loaded and configured with a custom ChatML chat template. The template includes TRL generation markers (`{% generation %}` and `{% endgeneration %}`) around assistant responses. These markers tell SFTTrainer exactly which tokens are the target output, enabling the `assistant_only_loss` feature so the model only learns to predict the assistant response and not the prompt tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# override with a TRL-compatible ChatML template that includes generation markers
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "<|im_start|>assistant\n{% generation %}{{ message['content'] }}{% endgeneration %}<|im_end|>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)

print(f"pad_token: {tokenizer.pad_token}")
print(f"vocab size: {tokenizer.vocab_size}")

### 8d: Dataset Loading and Splitting

The training data is loaded from JSON and split into train (80%), validation (10%) and test (10%) sets with a fixed random seed for reproducibility. The test set is saved to disk as a quarantined JSONL file that is never used during training, preserving it for unbiased evaluation later. The formatting function from Step 5 is applied via `.map()` to convert raw examples into the chat message format. Original columns are dropped after transformation to free memory.

In [ ]:
data_path = "./mnt/data/train.json"
dataset = load_dataset("json", data_files=data_path, split="train")

# 80/10/10 split
split_1 = dataset.train_test_split(test_size=0.2, seed=42)
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split_1["train"]
val_dataset = split_2["train"]
test_dataset = split_2["test"]

print(f"train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}")

# quarantine the test set so it is never touched during training
if local_rank == 0:
    print("saving quarantined test set to ./mnt/data/test_split.jsonl...")
    test_dataset.to_json("./mnt/data/test_split.jsonl")

# apply the chat template transformation to train and val sets
print("formatting dataset for the specific model...")

train_dataset = train_dataset.map(
    lambda row: format_messages_for_training(row, model_name), 
    remove_columns=["user_message", "intent", "session_history"]
)

val_dataset = val_dataset.map(
    lambda row: format_messages_for_training(row, model_name), 
    remove_columns=["user_message", "intent", "session_history"]
)
# remove_columns drops the original fields after they have been folded
# into the 'messages' column, freeing memory

print(f"train columns after formatting: {train_dataset.column_names}")
print(f"sample formatted row:")
print(json.dumps(train_dataset[0], indent=2))
    

### 8e: LoRA Configuration

LoRA (Low Rank Adaptation) is configured with rank 16 and alpha 32 (2x the rank, a common heuristic). The adapter targets all attention projections (q, k, v, o) and the MLP layers (gate, up, down) of the transformer. This allows fine tuning only a small fraction of the total parameters while keeping the base weights frozen.

In [ ]:
r = 16
peft_config = LoraConfig(
    r=r,
    lora_alpha=(2 * r),
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

### 8f: Training Arguments (SFTConfig)

Key training hyperparameters include:

- max_length 1024: maximum sequence length for tokenization
- batch size 1 with 4 gradient accumulation steps: effective batch size of 4
- learning rate 2e-4 with cosine scheduler and 10% warmup
- 3 training epochs with evaluation every 10% of steps
- fp16 mixed precision (bf16 is explicitly disabled to avoid crashes on T4 GPUs)
- gradient checkpointing enabled to trade compute time for memory savings
- the best model checkpoint is selected based on lowest eval_loss
- `assistant_only_loss=True`: only the assistant (target) tokens contribute to the loss. this is critical for intent classification because the model should not be penalized for the prompt

In [ ]:
training_args = SFTConfig(
    output_dir=output_dir,
    max_length=1024,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=25,
    num_train_epochs=3,
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    #fp16=True,
    fp16=False,
    bf16=True,
    #bf16=False, # Explicitly deny bfloat16 to prevent T4 crashes
    push_to_hub=False,
    packing=False,
    assistant_only_loss=True, # <-- Critical for intent classification

    # mlflow tracking
    report_to="mlflow",
    run_name=run_name,

    # evaluation and checkpointing
    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy="steps",
    save_steps=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Memory optimizations
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}, 
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
)

In [ ]:

from types import SimpleNamespace
args_ns = SimpleNamespace(
    model_name=model_name,
    data_path=data_path,
    dataset_source=dataset_source
)

lineage_callback = MLflowLineageCallback(
    custom_args=args_ns,
    peft_config=peft_config,
    train_ds=train_dataset,
    val_ds=val_dataset,
    test_ds=test_dataset
)


### 8g: Trainer Setup and Execution

The SFTTrainer is initialized with the quantized model, LoRA config, formatted datasets, tokenizer and the custom MLflow lineage callback. Calling `trainer.train()` starts the supervised fine tuning loop.

In [ ]:

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config, 
    processing_class=tokenizer,
    callbacks=[lineage_callback]
)

print("starting training...")
trainer.train()